# Reproduce the paper from the shipped checkpoints

This notebook **re-runs inference** from the checkpoints under `checkpoints/` for every paper
item that fits in minutes on one GPU, and compares each recomputed number with the number it
should reproduce. Nothing is read back from a stored result and passed off as a rerun:
stored results appear only in the *expected* column. **No training happens here.**

**Corrected configurations.** Where the audit (`docs/STATE_OF_CLAIMS.md`) found that the
published configuration was wrong, this notebook runs the corrected one and compares against
the corrected expected number, not the published one. The published value is shown beside it
so the change is visible.

| item | what changes | expected (corrected) |
|---|---|---|
| Table 1 / Fig 3 | dMFM at **1 composed step x MC 32** (NFE 32), scale-free gradient rule; at 1 step the step-matched-student rule selects the **diagonal** student. The published 4 x 8 split was GLASS's estimator. | guided oracle MAE **0.153 / 0.117** (n=1000) |
| Table 13 | the **scale-free** gradient rule (the 1 x 8 split was already optimal) | paired reduction **0.139** (n=100), was 0.126 |
| Table 16 | **total-variation distance** with a real-vs-real floor; the Pearson rows were noise | floor 0.0101, generated 0.0144, uniform 0.1064 |
| Table 18 | report **cosine similarity**, not MAE. The GLASS-128 reference has gradient norm ~0.01 against estimates of 0.2-10, so the MAE measures gradient size (corr 0.995), not accuracy. Cosine is what the scale-free guidance step uses. | per-config cosine of the stored 32 x 4 grid |
| Table 2 (QM9) | none: the published 4 x 32 split is already the optimum; SS uses Algorithm 3's MFM look-aheads | 3-seed n=1000 means |

**How to read the comparison tables.** Every recomputed number goes through `compare()` from
`_common.py` (via `R.check`, which calls it once per row with that row's own tolerance and
records why in `tol_basis`). Deterministic quantities (data statistics, unguided samples,
calibration) must agree to a tight absolute tolerance. Guided sampling is **not
bit-reproducible** (TF32 matmuls, nondeterministic CUDA backward), so guided quantities are
judged against the 95% CI of the small-n estimate. Where the stored full-size run used the same
seeds, the same sample ids are compared first ("same ids / same cells"); that is the sharp test.
A `MISMATCH` is left as it is and explained; nothing is tuned to agree.

**CPU vs GPU.** Table 11, Table 16 and Tables 4-5 run on CPU (Table 16 at a smaller n). Every
other section needs CUDA and prints a SKIPPED line without it, so the notebook executes end to end
on a login node. Outputs go to `outputs/reproduce_from_checkpoints/` (gitignored).

**Not run here, and why** (the heavy ones) is listed in the last section.

In [1]:
# Knobs. GPU sections run when CUDA is available; set RUN_GPU = False to force the CPU-only subset.
RUN_GPU = None
from _common import *

import scipy.linalg  # noqa: F401  load scipy (and the env's libstdc++) before torch: the kernel may start without scripts/env.sh
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

import _repro as R

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

GPU = torch.cuda.is_available() if RUN_GPU is None else (bool(RUN_GPU) and torch.cuda.is_available())
DEV = "cuda:0" if GPU else "cpu"

# Small-n settings (minutes each on one A100). Full-n commands are in each section's markdown.
N = {
    "table16_gpu": 4096, "table16_cpu": 256,   # generated sequences (4,096 = the stored headline set)
    "table1": 64,                              # guided/unguided pairs per target (stored: 1,000)
    "table13": 48,                             # pairs (stored: 100)
    "table18": (8, 2),                         # probes x repeats per reward (stored: 32 x 4)
    "fig5": (4, 8),                            # probe states x noise pools (published: 32 x 8)
    "table17": 8,                              # source states (published: 32)
    "qm9": 64,                                 # molecules per property and row (stored: 1,000 x 3 seeds)
}

SCORES, TIMES, SKIPPED = [], {}, []
T_START = time.time()


def show(df, section):
    """Display a comparison frame and keep it for the final summary."""
    SCORES.append(df.assign(section=section))
    n_mis = int((df.status == "MISMATCH").sum())
    print(f"{section}: {int((df.status == 'match').sum())} match, {n_mis} MISMATCH")
    display(df)


ERRORS = []


def attempt(section, fn, *args, **kw):
    """Run one section; on an exception print the traceback, record it, and let the notebook go on
    (the summary lists every errored section, so a failure is never silent)."""
    import traceback

    t0 = time.time()
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        traceback.print_exc()
        ERRORS.append((section, f"{type(exc).__name__}: {exc}"))
        print(f"ERROR in {section}; continuing with the next section.")
        return None
    finally:
        TIMES[section] = time.time() - t0


def need_gpu(section):
    if GPU:
        return True
    msg = f"SKIPPED {section}: needs a CUDA GPU (none visible here). Run this notebook on a GPU node."
    SKIPPED.append((section, "no CUDA device in this session"))
    print(msg)
    return False


print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", R.gpu_name() or "CPU only",
      "| GPU sections:", "ON" if GPU else "OFF")
print("host:", os.uname().nodename, "| Slurm job:", os.environ.get("SLURM_JOB_ID", "none"))

ver = R.verify_checkpoints()
print(f"checkpoint manifest: {ver['n_ok']} artifacts ok; {ver['last_line']}")
assert ver["returncode"] == 0 and not ver["problems"], ver
REPO

torch 2.5.1+cu124 | CUDA: True | NVIDIA A100-SXM4-40GB | GPU sections: ON
host: holygpu8a19206.rc.fas.harvard.edu | Slurm job: 49533348


checkpoint manifest: 59 artifacts ok; all artifacts verified


PosixPath('/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM')

## Table 11: C0 guide and oracle regressors (CPU)

Reproduces the held-out accuracy of the two shipped C0 regressors
(`checkpoints/dna/c0/{guide,oracle}`) on the 58 test parents (8,294 windows), reverse-complement
averaged as in training. Deterministic, so it must match the paper **to the printed digit**:
guide MAE 0.1588 / RMSE 0.2306 / r 0.8774 / rho 0.8838, oracle 0.1620 / 0.2347 / 0.8724 / 0.8764.
n = 8,294 windows (the full test set; this is already full size).

In [2]:
res = attempt("Table 11", R.run_table11)
if res is not None:
    show(res[0], "Table 11")

Table 11: 8 match, 0 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,Table 11 guide mae,0.1588,0.158771,match,0.00005,printed,8294,paper value; deterministic forward pass
1,Table 11 guide rmse,0.2306,0.230643,match,0.00005,printed,8294,paper value; deterministic forward pass
2,Table 11 guide pearson,0.8774,0.877414,match,0.00005,printed,8294,paper value; deterministic forward pass
3,Table 11 guide spearman,0.8838,0.883830,match,0.00005,printed,8294,paper value; deterministic forward pass
4,Table 11 oracle mae,0.1620,0.161985,match,0.00005,printed,8294,paper value; deterministic forward pass
5,Table 11 oracle rmse,0.2347,0.234695,match,0.00005,printed,8294,paper value; deterministic forward pass
6,Table 11 oracle pearson,0.8724,0.872416,match,0.00005,printed,8294,paper value; deterministic forward pass
7,Table 11 oracle spearman,0.8764,0.876359,match,0.00005,printed,8294,paper value; deterministic forward pass


## Table 16 (corrected): total-variation distance of base-composition marginals (CPU, GPU for full n)

The published Table 16 reported Pearson r on four constrained frequencies; against a sampled
uniform reference that r is exactly Uniform(-1, 1), i.e. noise. The corrected table
(`scripts/dna/table16_tv.py`, manuscript commit 7d5e375) reports total-variation distance with a
**real-vs-real floor**:

| comparison | TV (expected) |
|---|---|
| real (train parents) vs real (held-out parents), the floor | **0.0101** |
| generated vs real (held-out parents) | **0.0144** |
| uniform vs real (held-out parents) | **0.1064** |

Here the generated set is **re-sampled** from the L=50 base DFM (`checkpoints/dna/base/L50`)
with the stored run's seeded noise (128 Euler steps to t=1). On GPU n = 4,096, the stored headline
set, so every row should match exactly (the noise is seeded on CPU and the decode margin is ~1).
On CPU n = 256 (about 2 min on one core): the deterministic rows (floor, uniform) must still match
exactly, and the generated rows are judged against twice their bootstrap SD.
Full n (10,000 samples): `sbatch scripts/dna/regen/fig6_table16_marginals.sbatch`,
`bash scripts/dna/regen/collect.sh`, then `python scripts/dna/table16_tv.py`.

In [3]:
n16 = N["table16_gpu"] if GPU else N["table16_cpu"]
res = attempt("Table 16", R.run_table16, n16, DEV)
if res is not None:
    df16, info16 = res
    print(f"generated n={n16} on {DEV} (sampling {info16['seconds_sampling']:.0f}s)")
    display(info16["table"].round(6))
    show(df16, "Table 16 (TV)")

generated n=4096 on cuda:0 (sampling 14s)


,comparison,total_variation
0,FLOOR: train real vs held-out real,0.010084
1,generated vs held-out real,0.014430
2,uniform vs held-out real,0.106438
3,generated vs real (matched n),0.014834
4,uniform vs real (matched n),0.106416
5,generated vs training-split real,0.007395


Table 16 (TV): 9 match, 0 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,TV FLOOR: train real vs held-out real,0.010084,0.010084,match,0.000005,deterministic (data + seeds): 5e-6,8294 vs 32890,stored n=4096 run (results/dna/base_model/table16_tv.csv)
1,TV FLOOR: train real vs held-out real (as printed),0.0101,0.010084,match,0.000050,printed precision,8294 vs 32890,corrected manuscript Table 16 (commit 7d5e375)
2,TV generated vs held-out real,0.01443,0.014430,match,0.000050,"deterministic (seeded noise, decode margin ~1): 5e-5",8294 vs 4096,stored n=4096 run (results/dna/base_model/table16_tv.csv)
3,TV generated vs held-out real (as printed),0.0144,0.014430,match,0.000050,"deterministic (seeded noise, decode margin ~1): 5e-5",8294 vs 4096,corrected manuscript Table 16 (commit 7d5e375)
4,TV uniform vs held-out real,0.106438,0.106438,match,0.000005,deterministic (data + seeds): 5e-6,8294 vs 4096,stored n=4096 run (results/dna/base_model/table16_tv.csv)
5,TV uniform vs held-out real (as printed),0.1064,0.106438,match,0.000050,printed precision,8294 vs 4096,corrected manuscript Table 16 (commit 7d5e375)
6,TV generated vs real (matched n),0.014834,0.014834,match,0.000050,"deterministic (seeded noise, decode margin ~1): 5e-5",4096 vs 4096,stored n=4096 run (results/dna/base_model/table16_tv.csv)
7,TV uniform vs real (matched n),0.106416,0.106416,match,0.000005,deterministic (data + seeds): 5e-6,4096 vs 4096,stored n=4096 run (results/dna/base_model/table16_tv.csv)
8,TV generated vs training-split real,0.007395,0.007395,match,0.000050,"deterministic (seeded noise, decode margin ~1): 5e-5",32890 vs 4096,stored n=4096 run (results/dna/base_model/table16_tv.csv)


## Tables 4-5: diversity of the corrected Table 1 sequence sets (CPU)

Table 4 (real-data baselines) is posterior-independent; Table 5 measures the diversity of the
n = 1,000 C0-guided sets. The recompute uses the **corrected** Table 1 sets (dMFM 1 x 32, scale-free,
`results/dna/c0_guidance/dmfm_s1mc32_n1000/`), so Tables 1 and 4-5 describe the same samples.
This is CPU post-processing of shipped sequences (no generative inference; the GPU Table 1 section
below re-generates sequences from the checkpoints), at the full n = 1,000, a few minutes on one core.

Expected: Table 4 matches the paper where the split definition is unchanged; rows on the
parent-disjoint `a_train` / `a_val` / `test` splits differ in the third decimal because the
published table used the purged `split65k` sets. Table 5's +1 set is expected to keep
held-out-level diversity (published pairwise ratio 0.9997, nearest-generated 0.9447); its -1
claim ("measurably more clustered", 0.9507 / 0.8027) **did not reproduce** in the 4 x 8 dMFM
recompute (0.9927 / 0.9621) and is an open item (`docs/dna_findings_audit.md`), so MISMATCH rows
there are expected. The stored 4 x 8 recompute is shown in the `note` column.
Command: `python -m dmfm.experiments.analyze_c0_diversity --run_dirs results/dna/c0_guidance/dmfm_s1mc32_n1000/target_m1p0 results/dna/c0_guidance/dmfm_s1mc32_n1000/target_1p0 --out_dir <dir>`.

In [4]:
res = attempt("Tables 4-5", R.run_tables4_5)
if res is not None:
    df4, df5, info45 = res
    print(f"Tables 4-5 recomputed in {info45['seconds']:.0f}s")
    show(df4, "Table 4")
    show(df5, "Table 5")

                                                                                               run_dir  target_c0      set  n_fasta  n_sample_scores_csv  n_oracle_csv  fasta_equals_csv_in_order seq_lengths  n_exact_unique  n_samples_args_json  seed_args_json fasta_source
/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/results/dna/c0_guidance/dmfm_s1mc32_n1000/target_m1p0       -1.0   guided     1000                 1000          1000                       True          50            1000                 1000               0          csv
/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/results/dna/c0_guidance/dmfm_s1mc32_n1000/target_m1p0       -1.0 unguided     1000                 1000          1000                       True          50            1000                 1000               0          csv
 /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/results/dna/c0_guidance/dmfm_s1mc32_n1000/target_1p0        1.0   guided     1000                 1000          1000                  

{
  "split_sanity": {
    "a_train": {
      "n_windows": 32890,
      "n_parents": 230
    },
    "a_val": {
      "n_windows": 4147,
      "n_parents": 29
    },
    "b_train": {
      "n_windows": 32890,
      "n_parents": 230
    },
    "b_val": {
      "n_windows": 4147,
      "n_parents": 29
    },
    "test": {
      "n_windows": 8294,
      "n_parents": 58
    },
    "parent_sets_pairwise_disjoint": true
  },
  "token_order": {
    "n_unguided_sequences_used": 1000,
    "data_token_fraction_0123": [
      0.3103887432012432,
      0.19492229992229992,
      0.19330055361305362,
      0.30138840326340327
    ],
    "unguided_fasta_base_fraction_ACGT": {
      "A": 0.31736,
      "C": 0.19724,
      "G": 0.1915,
      "T": 0.2939
    },
    "best_mapping_by_trinuc_L1": "ACGT",
    "trinuc_L1_best_and_second": [
      0.04448435476560476,
      0.12889162457912456
    ],
    "second_best_mapping_by_trinuc_L1": "AGCT",
    "best_mapping_by_dinuc_L1": "ACGT",
    "dinuc_L1_best_and_

{
  "oracle_rescore_check": {
    "target_m1p0/guided": {
      "max_abs_diff_vs_stored": 5.888366705342207e-08,
      "rescored_mean": -0.9637488722801208
    },
    "target_m1p0/unguided": {
      "max_abs_diff_vs_stored": 5.5480956939746306e-08,
      "rescored_mean": -0.17146508395671844
    },
    "target_1p0/guided": {
      "max_abs_diff_vs_stored": 5.952453618895959e-08,
      "rescored_mean": 0.9202496409416199
    },
    "target_1p0/unguided": {
      "max_abs_diff_vs_stored": 5.5480956939746306e-08,
      "rescored_mean": -0.17146508395671844
    }
  }
}
# Tables 6/7 recomputed on the parent-disjoint model (auto-generated)

Normalized Hamming fraction = mismatches / L (L = 50). train = a_train (generator training parents),
test = held-out test parents, held-out = the 1,000-window test_sample baseline.

## Table 6: real-data baselines (1,000 windows, seed 0)

| set | n | exact unique | nearest generated | pairwise generated | pairwise median |
| --- | --- | --- | --- | --- | 

,item,expected,ours,status,tol,tol_basis,n,note
0,Table 4 train_sample exact_unique,1.0000,1.000000,match,0.00005,decimals,1000,paper; ours = parent-disjoint full_sample (posterior-independent)
1,Table 4 train_sample nearest_generated,0.5219,0.521880,match,0.00005,decimals,1000,paper; ours = parent-disjoint full_sample (posterior-independent)
2,Table 4 train_sample pairwise_generated,0.7377,0.737699,match,0.00005,decimals,1000,paper; ours = parent-disjoint full_sample (posterior-independent)
3,Table 4 train_sample pairwise_median,0.7400,0.740000,match,0.00005,decimals,1000,paper; ours = parent-disjoint full_sample (posterior-independent)
4,Table 4 split_train_sample exact_unique,1.0000,1.000000,match,0.00005,decimals,1000,paper; ours = parent-disjoint a_train_sample (posterior-independent)
5,Table 4 split_train_sample nearest_generated,0.5219,0.520000,MISMATCH,0.00005,decimals,1000,paper; ours = parent-disjoint a_train_sample (posterior-independent)
6,Table 4 split_train_sample pairwise_generated,0.7373,0.736074,MISMATCH,0.00005,decimals,1000,paper; ours = parent-disjoint a_train_sample (posterior-independent)
7,Table 4 split_train_sample pairwise_median,0.7400,0.740000,match,0.00005,decimals,1000,paper; ours = parent-disjoint a_train_sample (posterior-independent)
8,Table 4 split_val_sample exact_unique,1.0000,1.000000,match,0.00005,decimals,1000,paper; ours = parent-disjoint a_val_sample (posterior-independent)
9,Table 4 split_val_sample nearest_generated,0.5184,0.519360,MISMATCH,0.00005,decimals,1000,paper; ours = parent-disjoint a_val_sample (posterior-independent)


Table 5: 0 match, 10 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,Table 5 -1_guided guide_score_mean,-0.9893,-1.001524,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = -0.9390"
1,Table 5 -1_guided nearest_gen,0.4175,0.496260,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.5021"
2,Table 5 -1_guided pairwise_gen,0.7010,0.731447,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.7327"
3,Table 5 -1_guided pairwise_over_heldout,0.9507,0.990966,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9927"
4,Table 5 -1_guided nearest_gen_over_heldout,0.8027,0.950835,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9621"
5,Table 5 +1_guided guide_score_mean,1.0010,1.000368,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9817"
6,Table 5 +1_guided nearest_gen,0.4913,0.509460,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.5067"
7,Table 5 +1_guided pairwise_gen,0.7372,0.735990,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.7355"
8,Table 5 +1_guided pairwise_over_heldout,0.9997,0.997121,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9964"
9,Table 5 +1_guided nearest_gen_over_heldout,0.9447,0.976127,MISMATCH,0.00005,decimals,1000,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9708"


## Table 1 / Fig 3 (corrected): C0-guided generation with dMFM at 1 step x MC 32 (GPU)

Reproduces the paired guided-vs-unguided C0 experiment at L = 50 for y* = +1 and y* = -1, at the
**corrected operating point**: value gradients from **one composed flow-map step x MC 32** (NFE 32)
of the **diagonal** dMFM student (`checkpoints/dna/dmfm/L50`, picked by the step-matched-student
rule), **scale-free** gradient rule, every other flag as the stored run
(`dmfm.experiments.sample_c0_guidance_dmfm --paired --posterior dmfm --grad_normalize --nfe_value 1 --mc 32`),
scored with the held-out oracle.

Expected (n = 1,000): guided oracle MAE **0.153 [0.146, 0.161]** (+1) and **0.117 [0.111, 0.123]** (-1),
from unguided 1.174 / 0.833, 98.3% / 97.2% of pairs improved. (Published: 0.157 / 0.153 from an
n = 32 GLASS pilot; the 4 x 8 dMFM split gave 0.193 / 0.172.)
n used: 64 pairs per target. Sample i uses seed i, so these are the first 64 pairs of the stored
n = 1,000 run: the unguided arm must match exactly, the guided arm within its CI.
Full n: `NFE_VALUE=1 MC=32 sbatch scripts/dna/table1_c0_guidance_dmfm.sbatch`, then collect with
`python -m dmfm.experiments.collect_table1_c0 --run_root outputs/dna/table1_dmfm4_norm_s1mc32_n1000 --results_dir <dir> --summary_json <file> --n_samples 1000 --jobs <jobid>`
(do not use `collect_table1_dmfm.sh` for this tree: its `RUN_ROOT` defaults to the 4 x 8 run).

In [5]:
if need_gpu("Table 1 / Fig 3"):
    res = attempt("Table 1", R.run_table1, N["table1"], DEV)
if GPU and res is not None:
    df1, info1 = res
    show(df1, "Table 1")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for ax, (key, d) in zip(axes, info1["per_sample"].items()):
        bins = np.linspace(-2.0, 2.0, 41)
        ax.hist(d.oracle_unguided, bins=bins, alpha=0.6, label="unguided")
        ax.hist(d.oracle_guided, bins=bins, alpha=0.6, label="guided (dMFM 1x32)")
        ax.axvline(float(key), color="k", ls="--", lw=1)
        ax.set(title=f"y* = {key}, n = {len(d)}", xlabel="oracle C0")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    print("saved", savefig(fig, "reproduce_fig3_oracle_hist.png"))
    display(fig)
    plt.close(fig)

device=cuda protocol=paired/dmfm target=[1.0] n=64 mc=32 out_dir=/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table1/target_1p0


progress: 8/64


progress: 16/64


progress: 24/64


progress: 32/64


...

wrote /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table1/target_m1p0/sample_scores.csv


                                                                                                            run_dir  target_c0                    set  n_samples  oracle_score_mean  oracle_target_mae  oracle_target_mae_ci95_low  oracle_target_mae_ci95_high  fraction_guided_improved
/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table1/target_m1p0       -1.0               unguided         64          -0.190232           0.809768                    0.726947                     0.895047                       NaN
/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table1/target_m1p0       -1.0                 guided         64          -0.950052           0.116666                    0.096887                     0.137880                       NaN
/n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table1/target_m1p0       -1.0 paired_guidance_effect     

,item,expected,ours,status,tol,tol_basis,n,note
0,Table 1 y*=+1 unguided oracle MAE,1.190650,1.190705,match,0.001000,unguided arm is deterministic: 1e-3,64,"stored n=1000 run, same 64 sample ids"
1,Table 1 y*=+1 guided oracle MAE (same ids),0.165539,0.169049,match,0.026172,95% CI half-width of ours (n=64),64,"stored n=1000 run, same ids; paired diff +0.0035 +- 0.0188 (SE)"
2,Table 1 y*=+1 guided oracle MAE (n=1000 headline),0.153000,0.169049,match,0.026172,95% CI half-width of ours (n=64),64,"corrected Table 1 cell, 1 step x MC 32 (published 4x8 split gave 0.157 n=32 pilot / 0...."
3,Table 1 y*=+1 fraction of pairs improved,0.983000,0.984375,match,0.034300,binomial 95% half-width at p=0.98,64,n=1000 run
4,Table 1 y*=-1 unguided oracle MAE,0.809822,0.809768,match,0.001000,unguided arm is deterministic: 1e-3,64,"stored n=1000 run, same 64 sample ids"
5,Table 1 y*=-1 guided oracle MAE (same ids),0.135546,0.116666,match,0.020497,95% CI half-width of ours (n=64),64,"stored n=1000 run, same ids; paired diff -0.0189 +- 0.0169 (SE)"
6,Table 1 y*=-1 guided oracle MAE (n=1000 headline),0.117000,0.116666,match,0.020497,95% CI half-width of ours (n=64),64,"corrected Table 1 cell, 1 step x MC 32 (published 4x8 split gave 0.153 n=32 pilot / 0...."
7,Table 1 y*=-1 fraction of pairs improved,0.972000,1.000000,match,0.034300,binomial 95% half-width at p=0.98,64,n=1000 run


saved /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/notebooks/figures/diagnostics/reproduce_fig3_oracle_hist.png


<Figure size 1000x320 with 2 Axes>

## Table 13 (corrected rule): exact motif-reward attainment (GPU)

Paired reduction in exactly computed target error for the motif reward at L = 50, guided with one
direct diagonal-dMFM map x MC 8 (the only split `sample_reward_attainment` has, and the optimum
at this budget), under the **scale-free** rule (corrected) and the published clip-down rule.
Run through `dmfm.experiments.operating_point_reward_grid --steps_values 1 --nfe 8`, which is the
published configuration plus the rule switch.

Expected: scale-free **0.139 [0.120, 0.160]** (stored n = 100, H200); clip-down 0.126 [0.109, 0.144]
(published, reproduced as 0.1262 by the stored run). n used: 48 pairs (the first 48 sample ids of
the stored run, so the unguided arm and the 1,024-sample target calibration must match exactly).
Full n and the other lengths/rewards: `sbatch scripts/dna/operating_point/table13_grid.sbatch`
(and `scripts/dna/table13_reward_attainment.sbatch` for the published clip-down table).

In [6]:
if need_gpu("Table 13"):
    res = attempt("Table 13", R.run_table13, N["table13"], DEV)
    if res is not None:
        show(res[0], "Table 13")

L=50 motif: target=0.587551 scale=0.105798


  dmfm_fm1@clip@mc=8                 guided|err|=0.0246 paired=+0.1317 |g|raw=1.56 (6s)


  dmfm_fm1@norm@mc=8                 guided|err|=0.0058 paired=+0.1506 |g|raw=0.818 (5s)


Table 13: 8 match, 0 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,Table 13 calibrated target score (90th pct),0.587551,0.587551,match,0.001000,deterministic calibration: 1e-3,1024,"stored GPU run, Slurm 49479325"
1,Table 13 motif L50 scale-free (corrected): unguided |err|,0.156333,0.156333,match,0.001000,unguided arm is deterministic: 1e-3,48,"stored run, same sample ids"
2,Table 13 motif L50 scale-free (corrected): paired reduction (same ids),0.142022,0.150552,match,0.026559,95% CI half-width of ours (n=48),48,stored n=100 run restricted to the same sample ids
3,Table 13 motif L50 scale-free (corrected): paired reduction (n=100),0.139253,0.150552,match,0.026559,95% CI half-width of ours (n=48),48,stored n=100 GPU run (results/dna/operating_point/table13_grid/motif_nfe8)
4,Table 13 motif L50 clip-down (published rule): unguided |err|,0.156333,0.156333,match,0.001000,unguided arm is deterministic: 1e-3,48,"stored run, same sample ids"
5,Table 13 motif L50 clip-down (published rule): paired reduction (same ids),0.126021,0.131715,match,0.024983,95% CI half-width of ours (n=48),48,stored n=100 run restricted to the same sample ids
6,Table 13 motif L50 clip-down (published rule): paired reduction (n=100),0.126223,0.131715,match,0.024983,95% CI half-width of ours (n=48),48,stored n=100 GPU run (results/dna/operating_point/table13_grid/motif_nfe8)
7,Table 13 motif L50 clip-down (published rule): paired reduction (paper),0.126,0.131715,match,0.024983,95% CI half-width of ours (n=48),48,"published Table 13 cell, 0.126 [0.109, 0.144]"


## Table 18 (corrected metric): value-gradient cosine similarity with the GLASS-128 reference (GPU)

Table 18 printed the per-coordinate MAE between the dMFM finite-MC value gradient and a GLASS-128
(RK4, 50 steps, t_end 0.999) reference. That reference is numerically degenerate (L2 norm
~0.005-0.09 against estimates of 0.2-10), so the MAE tracks the estimate's own norm
(corr 0.95 on `gc`, 0.995 on `motif`) rather than its accuracy. The corrected table reports the
**cosine similarity**, the quantity the scale-free guidance step actually uses (it normalises the
magnitude away). Rows: the published configuration (4-step student, 4 composed steps, MC 1/2/4/8),
the corrected operating point at NFE 32 (1 step x MC 32, diagonal student; the 4-step student at
1 step for reference) and GLASS 4 x 8 on the base model.

Expected: the stored GPU grid (`results/dna/operating_point/table18_grid/<reward>/L50`, 32 probes x
4 repeats): `gc` cos 0.137 / 0.142 / 0.128 / 0.112 at 4 x {1, 2, 4, 8} and **0.297** at 1 x 32
(diagonal); `motif` 0.170 / 0.132 / 0.146 / 0.128 and **0.302**. n used: 8 probes x 2 repeats per
reward, seeded exactly as the stored grid, so they are the same 16 cells (compared first) and a
subset of the 128.
Full n: `sbatch scripts/dna/operating_point/table18_grid.sbatch` (or
`python -m dmfm.experiments.operating_point_gradient_grid --length 50 --reward gc --output_dir <dir>`).

In [7]:
if need_gpu("Table 18"):
    for reward in ("gc", "motif"):
        res = attempt(f"Table 18 {reward}", R.run_table18, reward, *N["table18"], DEV)
        if res is None:
            continue
        df18, info18 = res
        show(df18, f"Table 18 cos ({reward})")
        lo, hi = info18["reference_l2_range"]
        print(f"{reward}: reference |grad| range {lo:.4f}-{hi:.4f}; corr(MAE, |estimate|) over all cells = "
              f"{info18['corr_mae_vs_estimate_norm']:.3f}  <- why MAE is not reported")
        display(info18["diag"].round(4))

L=50 gc: probe 1/8 (23s)


L=50 gc: probe 2/8 (47s)


L=50 gc: probe 3/8 (70s)


L=50 gc: probe 4/8 (94s)


L=50 gc: probe 5/8 (117s)


...

wrote /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table18/motif/L50/gradient_grid.csv


Table 18 cos (motif): 13 match, 1 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 1 (NFE 4) [same cells]",0.203094,0.193417,match,0.092767,95% CI half-width of ours (16 cells),16,"stored GPU grid, same probes/repeats"
1,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 1 (NFE 4) [32x4 grid]",0.170481,0.193417,match,0.092767,95% CI half-width of ours (16 cells),16,"stored GPU grid, all 32 probes x 4 repeats (results/dna/operating_point/table18_grid)"
2,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 2 (NFE 8) [same cells]",0.089000,0.080238,match,0.105027,95% CI half-width of ours (16 cells),16,"stored GPU grid, same probes/repeats"
3,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 2 (NFE 8) [32x4 grid]",0.131849,0.080238,match,0.105027,95% CI half-width of ours (16 cells),16,"stored GPU grid, all 32 probes x 4 repeats (results/dna/operating_point/table18_grid)"
4,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 4 (NFE 16) [same cells]",0.091311,0.080793,match,0.064253,95% CI half-width of ours (16 cells),16,"stored GPU grid, same probes/repeats"
5,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 4 (NFE 16) [32x4 grid]",0.146005,0.080793,MISMATCH,0.064253,95% CI half-width of ours (16 cells),16,"stored GPU grid, all 32 probes x 4 repeats (results/dna/operating_point/table18_grid)"
6,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 8 (NFE 32) [same cells]",0.139349,0.126605,match,0.080751,95% CI half-width of ours (16 cells),16,"stored GPU grid, same probes/repeats"
7,"Table 18 motif L50 cos, dMFM 4-step student, 4 steps, MC 8 (NFE 32) [32x4 grid]",0.128352,0.126605,match,0.080751,95% CI half-width of ours (16 cells),16,"stored GPU grid, all 32 probes x 4 repeats (results/dna/operating_point/table18_grid)"
8,"Table 18 motif L50 cos, dMFM diagonal student, 1 step, MC 32 (NFE 32) [same cells]",0.306235,0.293648,match,0.084415,95% CI half-width of ours (16 cells),16,"stored GPU grid, same probes/repeats"
9,"Table 18 motif L50 cos, dMFM diagonal student, 1 step, MC 32 (NFE 32) [32x4 grid]",0.301824,0.293648,match,0.084415,95% CI half-width of ours (16 cells),16,"stored GPU grid, all 32 probes x 4 repeats (results/dna/operating_point/table18_grid)"


motif: reference |grad| range 0.0056-0.0473; corr(MAE, |estimate|) over all cells = 0.996  <- why MAE is not reported


,config,mc,cos_mean,mae_mean,estimate_l2_mean,reference_l2_median
0,dmfm4_fm4,1,0.1934,0.8225,48.6599,0.0092
1,dmfm4_fm4,2,0.0802,0.8566,50.9467,0.0092
2,dmfm4_fm4,4,0.0808,0.3325,19.8467,0.0092
3,dmfm4_fm4,8,0.1266,0.1416,8.5934,0.0092
4,dmfm_fm1,32,0.2936,0.0632,2.7947,0.0092
5,dmfm4_fm1,32,0.2958,0.0480,1.6909,0.0092
6,glass_euler4,8,0.4065,0.0998,3.1200,0.0092


## Fig 5 / Tables 14-15: finite-N gradient error against a GLASS-2048 reference (GPU)

Scale-free value-gradient error (cosine with an independent GLASS-2048 reference) as a function of
the Monte Carlo budget N, `gc` reward, L = 50. **Caveat:** the published Fig 5 used GLASS on the base
DFM on *both* sides, although the caption and App. F.2 describe dMFM estimates; this section reruns
what was actually published (`rerun.fig5`). The dMFM version is
`sbatch scripts/dna/scalefree/fig5_dmfm_gradient_mc.sbatch` and has no complete result yet.

Expected: the published per-state cells for the same states (stored
`results/dna/scale_free_mc/gc/L50/gradient_scale_free_per_state.csv`), and Table 15's 32-state
cosine at L = 50: 0.14 / 0.16 / 0.19 / 0.24 / 0.32 / 0.43 / 0.54 / 0.64 / ... for N = 1 ... 256.
n used: the first 4 of the 32 probe states x all 8 noise pools (probe states and pools are seeded,
so these are the published states 0-3). The 4-state mean is compared with the 32-state value
within the spread over states.
Full n: `sbatch scripts/dna/fig5_tables14_15_gradient_mc.sbatch`, then `bash scripts/dna/fig5_tables14_15_rescore.sh`.

In [8]:
if need_gpu("Fig 5 / Tables 14-15"):
    res = attempt("Fig 5", R.run_fig5, *N["fig5"], DEV)
if GPU and res is not None:
    df5f, info5f = res
    show(df5f, "Fig 5 / Table 15 (gc, L50)")
    ps = info5f["per_state"]
    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    m = ps.groupby("mc").cosine.mean()
    ax.plot(m.index, m.values, "o-", label=f"rerun ({ps.state.nunique()} states)")
    pub = load_json("dna/paper_values.json")["table15_scale_free_full"]["gc"]
    ax.plot([int(k) for k in pub], [pub[k]["50"][1] for k in pub], "s--", label="published (32 states)")
    ax.set(xscale="log", xlabel="N (Monte Carlo samples)", ylabel="cosine vs GLASS-2048", title="gc, L = 50")
    ax.legend(frameon=False)
    print("saved", savefig(fig, "reproduce_fig5_cosine.png"))
    display(fig)
    plt.close(fig)

Loading L=50 checkpoint: /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/checkpoints/dna/base/L50/best.pt


L=50: gc calibration mean=0.383574, std=0.087726, center=0.383574


L=50: probe 1/4


L=50: probe 2/4


L=50: probe 3/4


...

Wrote /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/fig5/gradient_accuracy_relative_directional_summary.csv


Fig 5 / Table 15 (gc, L50): 12 match, 6 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,"Fig 5 gc L50 cosine, N=1 [same 4 states]",0.216487,0.216568,match,0.037448,1.96 x SD/sqrt(4) over states (floor 0.01),4,stored published-run per-state cells
1,"Fig 5 / Table 15 gc L50 cosine, N=1 [paper, 32 states]",0.14,0.216568,MISMATCH,0.037448,1.96 x SD/sqrt(4) over states (floor 0.01),4,published 32-state value; ours covers 4 states only
2,"Fig 5 gc L50 cosine, N=2 [same 4 states]",0.194519,0.194623,match,0.067709,1.96 x SD/sqrt(4) over states (floor 0.01),4,stored published-run per-state cells
3,"Fig 5 / Table 15 gc L50 cosine, N=2 [paper, 32 states]",0.16,0.194623,match,0.067709,1.96 x SD/sqrt(4) over states (floor 0.01),4,published 32-state value; ours covers 4 states only
4,"Fig 5 gc L50 cosine, N=4 [same 4 states]",0.259212,0.258846,match,0.034507,1.96 x SD/sqrt(4) over states (floor 0.01),4,stored published-run per-state cells
5,"Fig 5 / Table 15 gc L50 cosine, N=4 [paper, 32 states]",0.19,0.258846,MISMATCH,0.034507,1.96 x SD/sqrt(4) over states (floor 0.01),4,published 32-state value; ours covers 4 states only
6,"Fig 5 gc L50 cosine, N=8 [same 4 states]",0.290987,0.291023,match,0.064630,1.96 x SD/sqrt(4) over states (floor 0.01),4,stored published-run per-state cells
7,"Fig 5 / Table 15 gc L50 cosine, N=8 [paper, 32 states]",0.24,0.291023,match,0.064630,1.96 x SD/sqrt(4) over states (floor 0.01),4,published 32-state value; ours covers 4 states only
8,"Fig 5 gc L50 cosine, N=16 [same 4 states]",0.35483,0.354776,match,0.060839,1.96 x SD/sqrt(4) over states (floor 0.01),4,stored published-run per-state cells
9,"Fig 5 / Table 15 gc L50 cosine, N=16 [paper, 32 states]",0.32,0.354776,match,0.060839,1.96 x SD/sqrt(4) over states (floor 0.01),4,published 32-state value; ours covers 4 states only


saved /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/notebooks/figures/diagnostics/reproduce_fig5_cosine.png


<Figure size 450x320 with 1 Axes>

## Table 17: reward-free posterior diversity, one-step dMFM vs 100-step GLASS (GPU)

Normalised pairwise Hamming distance between 16 posterior futures per source state, for the
one-step diagonal dMFM map and the 100-step GLASS sampler, at t in {0, 0.25, 0.5, 0.75}, L = 50.
Expected (paper, 32 sources): dMFM / GLASS 0.732 / 0.736, 0.713 / 0.716, 0.563 / 0.555,
0.063 / 0.059. n used: the first 8 of the 32 source states (seeded permutation, so the same source
ids as the stored run, compared first). Full n and all lengths:
`sbatch scripts/dna/table17_posterior_diversity.sbatch`.

In [9]:
if need_gpu("Table 17"):
    res = attempt("Table 17", R.run_table17, N["table17"], DEV)
    if res is not None:
        show(res[0], "Table 17 (L50)")

L=50: sources=8 student=epoch=95-step=49000.ckpt


  t=0 method=dmfm complete


  t=0 method=glass complete


  t=0.25 method=dmfm complete


  t=0.25 method=glass complete


...

  t=0.75 method=dmfm complete


  t=0.75 method=glass complete


wrote /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/outputs/reproduce_from_checkpoints/gpu_49533348/table17


Table 17 (L50): 15 match, 1 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,Table 17 L50 t=0.00 dmfm pairwise Hamming [same sources],0.730958,0.731042,match,0.005000,"95% CI half-width of ours (8 sources, floor 0.005)",8,"stored run, same 8 source ids"
1,Table 17 L50 t=0.00 dmfm pairwise Hamming [paper],0.732,0.731042,match,0.005000,"95% CI half-width of ours (8 sources, floor 0.005)",8,published value (32 sources)
2,Table 17 L50 t=0.00 glass pairwise Hamming [same sources],0.735167,0.735167,match,0.005073,"95% CI half-width of ours (8 sources, floor 0.005)",8,"stored run, same 8 source ids"
3,Table 17 L50 t=0.00 glass pairwise Hamming [paper],0.736,0.735167,match,0.005073,"95% CI half-width of ours (8 sources, floor 0.005)",8,published value (32 sources)
4,Table 17 L50 t=0.25 dmfm pairwise Hamming [same sources],0.718854,0.718771,match,0.005896,"95% CI half-width of ours (8 sources, floor 0.005)",8,"stored run, same 8 source ids"
5,Table 17 L50 t=0.25 dmfm pairwise Hamming [paper],0.713,0.718771,match,0.005896,"95% CI half-width of ours (8 sources, floor 0.005)",8,published value (32 sources)
6,Table 17 L50 t=0.25 glass pairwise Hamming [same sources],0.721625,0.721625,match,0.005355,"95% CI half-width of ours (8 sources, floor 0.005)",8,"stored run, same 8 source ids"
7,Table 17 L50 t=0.25 glass pairwise Hamming [paper],0.716,0.721625,MISMATCH,0.005355,"95% CI half-width of ours (8 sources, floor 0.005)",8,published value (32 sources)
8,Table 17 L50 t=0.50 dmfm pairwise Hamming [same sources],0.566771,0.566812,match,0.008271,"95% CI half-width of ours (8 sources, floor 0.005)",8,"stored run, same 8 source ids"
9,Table 17 L50 t=0.50 dmfm pairwise Hamming [paper],0.563,0.566812,match,0.008271,"95% CI half-width of ours (8 sources, floor 0.005)",8,published value (32 sources)


## Table 2 (QM9): multiMFM and multiMFM-SS property guidance (GPU)

Six QM9 properties, the base TABASCO flow (`checkpoints/base_flow`) steered with the QM9 meta flow
map (`checkpoints/mfm_student`) at the **published 4 inner steps x 32 posterior samples**, which the
split sweep found to be already optimal (`docs/qm9_operating_point.md`), so the configuration is
unchanged. multiMFM: `scripts/qm9/run_multimfm.sh`; multiMFM-SS: `scripts/run_steer_search.sh` with
Algorithm 3's MFM look-aheads (`LOOKAHEAD_SOURCE=mfm`, all look-aheads kept, lumo mu 3.0), as in
the stored 3-seed reruns. Metric: oracle MAE (paper units) over all molecules; SS picks the best
PoseBusters-valid look-ahead.

Expected (3 seeds x n = 1,000): multiMFM Cv 0.444, mu 0.818, alpha 1.166, gap 592, HOMO 270,
LUMO 395; SS (Alg. 3) 0.414, 0.528, 1.009, 413, 243, 265. Published (seed 8): multiMFM 0.42 / 0.84 /
1.12 / 600 / 262 / 392; SS 0.40 / 0.69 / 1.10 / 462 / 252 / 316 (the published SS row used the
base-endpoint harvest, so the SS "paper" rows are not the target). n used: 64 molecules per
property and row, evaluation seed 1 (target seed 2027), i.e. the first 64 molecules (same lengths
and targets) of the stored seed-1 run, compared first.
Full n: `sbatch scripts/qm9/table2_3seed.sbatch`, then `bash scripts/qm9/collect_table2_3seed.sh`.

In [10]:
if need_gpu("Table 2 (QM9)"):
    res = attempt("Table 2 (QM9)", R.run_qm9, N["qm9"])
    if res is not None:
        df2, info2 = res
        if len(df2):
            show(df2, "Table 2 (QM9)")
        for kind, prop, code, log in info2["failures"]:
            ERRORS.append((f"Table 2 (QM9) {kind}/{prop}", f"exit {code}, log {log}"))
        print("seconds per run:", {f"{k[0]}/{k[1]}": round(v) for k, v in info2["seconds"].items()})

Table 2 (QM9): 34 match, 2 MISMATCH


,item,expected,ours,status,tol,tol_basis,n,note
0,Table 2 multiMFM cv [cal/(mol K)] (same molecules),0.457988,0.433836,match,0.110468,95% CI half-width of ours (n=64),64,"stored seed-1 n=1000 run, first 64 molecules; PB valid 0.891"
1,Table 2 multiMFM cv [cal/(mol K)] (3-seed n=1000),0.444352,0.433836,match,0.110468,95% CI half-width of ours (n=64),64,"3-seed mean, SE 0.00364"
2,Table 2 multiMFM cv [cal/(mol K)] (paper),0.42,0.433836,match,0.110468,95% CI half-width of ours (n=64),64,published (seed 8)
3,Table 2 multiMFM-SS (Alg. 3) cv [cal/(mol K)] (same molecules),0.412899,0.464845,match,0.097052,95% CI half-width of ours (n=64),64,"stored seed-1 n=1000 run, first 64 molecules; PB valid 1.000"
4,Table 2 multiMFM-SS (Alg. 3) cv [cal/(mol K)] (3-seed n=1000),0.41392,0.464845,match,0.097052,95% CI half-width of ours (n=64),64,"3-seed mean, SE 0.00852"
5,Table 2 multiMFM-SS (Alg. 3) cv [cal/(mol K)] (paper),0.40,0.464845,match,0.097052,95% CI half-width of ours (n=64),64,published (seed 8); published SS used the base-endpoint harvest
6,Table 2 multiMFM mu [D] (same molecules),0.741256,0.794807,match,0.234065,95% CI half-width of ours (n=64),64,"stored seed-1 n=1000 run, first 64 molecules; PB valid 0.875"
7,Table 2 multiMFM mu [D] (3-seed n=1000),0.818393,0.794807,match,0.234065,95% CI half-width of ours (n=64),64,"3-seed mean, SE 0.0329"
8,Table 2 multiMFM mu [D] (paper),0.84,0.794807,match,0.234065,95% CI half-width of ours (n=64),64,published (seed 8)
9,Table 2 multiMFM-SS (Alg. 3) mu [D] (same molecules),0.383527,0.512916,match,0.130867,95% CI half-width of ours (n=64),64,"stored seed-1 n=1000 run, first 64 molecules; PB valid 1.000"


seconds per run: {'multimfm/cv': 66, 'ss/cv': 52, 'multimfm/mu': 47, 'ss/mu': 50, 'multimfm/alpha': 46, 'ss/alpha': 46, 'multimfm/gap': 45, 'ss/gap': 49, 'multimfm/homo': 49, 'ss/homo': 51, 'multimfm/lumo': 41, 'ss/lumo': 48}


## Not run here, and why

| item | reason | how to run it |
|---|---|---|
| Table 3, GEOM-Drugs (and its Table 21 row) | Heaviest molecular run, and a small n cannot test it: the claimed effect (~0.003 Tanimoto) needs n = 4,096 x 3 seeds to resolve, and at that size it **does not survive** (gain -0.0003 +- 0.0007, `docs/geom_rescue_attempt.md`). | `sbatch scripts/geom/steer_table3.sbatch` |
| Fig 4 (value error vs N; NFE comparison) | 162 GPU units plus four tuned search baselines; hours of GPU. | `scripts/dna/fig4_all.sbatch`, `scripts/dna/fig4_collect.sh` |
| Table 6 (Evo 2 NLL) | Needs the separate `evo2` environment and the 7B Evo 2 model. | `scripts/dna/dmfm_demos/table6_evo2_dmfm.sbatch` |
| Fig 7 / Fig 8 (tilt ladder, single trajectory) | n = 1,000 per ladder point x 14 points; same sampler as Table 1, which is reproduced above. | `scripts/dna/operating_point/fig7_at_operating_point.sbatch`, `scripts/dna/dmfm_demos/fig8_trace_dmfm.sbatch` |
| Table 13 at L = 100-400 and the conjunction reward; Table 17 and Table 18 at L = 100-400; Fig 5 at the other lengths and rewards | Cost grows with L (L = 400 is the bulk of each full run); L = 50 is reproduced above. | the sbatch wrappers named in each section |
| Tables 7-10, 19-20 (training), toy Fig 1 / Table 12 | Training is out of scope; the toy has no printed numbers to reproduce (its notebook regenerates Fig 1 from `checkpoints/toy`). | `notebooks/01_toy_multimodal.ipynb`, `scripts/*/train_*.sbatch` |
| Tables 21-22 (QM9 PB validity / uniqueness) | Produced by the Table 2 runs; PB validity is printed in the Table 2 notes but not judged here. | the Table 2 commands |

## Summary

In [11]:
summary = pd.concat(SCORES, ignore_index=True) if SCORES else pd.DataFrame(columns=["section", "status"])
by = summary.groupby("section", sort=False).status.value_counts().unstack(fill_value=0)
display(by)
mis = summary[summary.status == "MISMATCH"]
print(f"{len(summary)} comparisons: {int((summary.status == 'match').sum())} match, {len(mis)} MISMATCH")
if len(mis):
    display(mis[["section", "item", "expected", "ours", "tol", "tol_basis", "note"]])
if SKIPPED:
    print("Skipped in this session (no GPU):", ", ".join(s for s, _ in SKIPPED))
if ERRORS:
    print("ERRORED sections (see their tracebacks above):")
    for sec, msg in ERRORS:
        print(f"  {sec}: {msg}")
else:
    print("No section raised an error.")
print("section wall-clock (s):", {k: round(v) for k, v in TIMES.items()})
print(f"total {time.time() - T_START:.0f}s on {R.gpu_name() or 'CPU'}")

status,MISMATCH,match
section,,
"Fig 5 / Table 15 (gc, L50)",6,12
Table 1,0,8
Table 11,0,8
Table 13,0,8
Table 16 (TV),0,9
Table 17 (L50),1,15
Table 18 cos (gc),1,13
Table 18 cos (motif),1,13
Table 2 (QM9),2,34


157 comparisons: 130 match, 27 MISMATCH


,section,item,expected,ours,tol,tol_basis,note
22,Table 4,Table 4 split_train_sample nearest_generated,0.5219,0.520000,0.000050,decimals,paper; ours = parent-disjoint a_train_sample (posterior-independent)
23,Table 4,Table 4 split_train_sample pairwise_generated,0.7373,0.736074,0.000050,decimals,paper; ours = parent-disjoint a_train_sample (posterior-independent)
26,Table 4,Table 4 split_val_sample nearest_generated,0.5184,0.519360,0.000050,decimals,paper; ours = parent-disjoint a_val_sample (posterior-independent)
27,Table 4,Table 4 split_val_sample pairwise_generated,0.7372,0.738451,0.000050,decimals,paper; ours = parent-disjoint a_val_sample (posterior-independent)
30,Table 4,Table 4 split_test_sample nearest_generated,0.5201,0.521920,0.000050,decimals,paper; ours = parent-disjoint test_sample (posterior-independent)
31,Table 4,Table 4 split_test_sample pairwise_generated,0.7374,0.738115,0.000050,decimals,paper; ours = parent-disjoint test_sample (posterior-independent)
33,Table 5,Table 5 -1_guided guide_score_mean,-0.9893,-1.001524,0.000050,decimals,"paper (split65k, GLASS); stored dMFM 4x8 recompute = -0.9390"
34,Table 5,Table 5 -1_guided nearest_gen,0.4175,0.496260,0.000050,decimals,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.5021"
35,Table 5,Table 5 -1_guided pairwise_gen,0.701,0.731447,0.000050,decimals,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.7327"
36,Table 5,Table 5 -1_guided pairwise_over_heldout,0.9507,0.990966,0.000050,decimals,"paper (split65k, GLASS); stored dMFM 4x8 recompute = 0.9927"


No section raised an error.
section wall-clock (s): {'Table 11': 2, 'Table 16': 14, 'Tables 4-5': 166, 'Table 1': 23, 'Table 13': 19, 'Table 18 gc': 189, 'Table 18 motif': 190, 'Fig 5': 175, 'Table 17': 5, 'Table 2 (QM9)': 590}
total 1375s on NVIDIA A100-SXM4-40GB
